# 03 Portfolio Allocation

This notebook allocates portfolio weights for the 25 stocks selected in Step 02. It does not perform new stock selection.

Methods included:

1. Equal Weight
2. Inverse Volatility
3. Markowitz-style Mean-Volatility Optimization
4. Risk Parity
5. CVaR optimization with Bootstrap scenarios
6. CVaR optimization with Monte Carlo scenarios

The Markowitz implementation is Markowitz-style mean-risk optimization using volatility as the risk penalty, rather than the classic mean-variance objective using variance directly.

CVaR Monte Carlo depends on a multivariate normal return assumption. Bootstrap CVaR is less parametric because it samples from historical return observations.

In [1]:
from pathlib import Path
import json
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, linprog

# ----------------------------------------------------------------------
# Project paths
# ----------------------------------------------------------------------
def find_project_root() -> Path:
    """Find the repository root from either a script run or notebook run."""
    candidates = []
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    candidates.append(Path.cwd())

    for start in dict.fromkeys(candidates):
        for candidate in [start, *start.parents]:
            if (candidate / "data" / "returns_matrix.parquet").exists():
                return candidate

    raise FileNotFoundError("Could not find project root containing data/returns_matrix.parquet.")


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------------------
# Allocation configuration
# ----------------------------------------------------------------------
TRADING_DAYS = None          # estimated from the actual returns index after loading data
LOOKBACK_YEARS = None        # None = use all available history; integer = trailing calendar-year lookback
MAX_WEIGHT = 0.10            # concentration cap per stock
MIN_WEIGHT = 0.00            # long-only
RISK_FREE_RATE = None        # fetched from FRED DGS3MO after loading data
RISK_FREE_RATE_SOURCE = "FRED DGS3MO"

# Markowitz-style mean-risk optimization: maximize annual_return - delta * annual_volatility.
# Use a dense log-spaced grid so the mean-volatility frontier is smooth enough to interpret.
MARKOWITZ_DELTA_GRID = np.logspace(start=-2, stop=2, num=25)[::-1].tolist()  # conservative -> aggressive

# CVaR configuration
N_SCENARIOS = 3000
CVAR_ALPHA = 0.95            # worst 5% tail
CVAR_RETURN_TRADEOFF = 1.0   # default balanced CVaR portfolio
CVAR_TRADEOFF_GRID = np.logspace(start=-2, stop=2, num=25).tolist()  # conservative -> aggressive
RANDOM_SEED = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Outputs will be written to: {OUTPUT_DIR}")

Project root: C:\Users\oakza\Documents\stock-selection-funnel
Outputs will be written to: C:\Users\oakza\Documents\stock-selection-funnel\outputs


In [2]:
# ----------------------------------------------------------------------
# Load selected stocks, returns, real trading calendar, and real risk-free rate
# ----------------------------------------------------------------------
def estimate_trading_days_per_year(index: pd.Index) -> int:
    dates = pd.to_datetime(index)
    counts = pd.Series(1, index=dates).groupby(dates.year).sum()
    full_years = counts[(counts.index > dates.min().year) & (counts.index < dates.max().year)]
    base = full_years if len(full_years) else counts
    return int(round(base.median()))


def fetch_fred_risk_free_rate(as_of_date, series_id="DGS3MO"):
    """Fetch latest available 3-month Treasury yield from FRED on or before as_of_date."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    fred = pd.read_csv(url)
    date_col = "DATE" if "DATE" in fred.columns else "observation_date"
    value_col = series_id if series_id in fred.columns else fred.columns[-1]
    fred = fred.rename(columns={date_col: "DATE", value_col: series_id})
    fred["DATE"] = pd.to_datetime(fred["DATE"])
    fred[series_id] = pd.to_numeric(fred[series_id], errors="coerce")
    fred = fred.dropna(subset=[series_id])
    fred = fred[fred["DATE"] <= pd.Timestamp(as_of_date)]
    if fred.empty:
        raise ValueError(f"No FRED {series_id} observations found on or before {as_of_date}.")
    row = fred.iloc[-1]
    return {
        "source": "FRED",
        "series_id": series_id,
        "description": "3-Month Treasury Constant Maturity Rate",
        "observation_date": row["DATE"].date().isoformat(),
        "annual_rate_percent": float(row[series_id]),
        "annual_rate_decimal": float(row[series_id]) / 100.0,
    }


def fetch_yahoo_irx_risk_free_rate(as_of_date):
    """Fallback: fetch Yahoo Finance ^IRX 13-week T-bill yield via chart endpoint."""
    end = int((pd.Timestamp(as_of_date) + pd.Timedelta(days=1)).timestamp())
    start = int((pd.Timestamp(as_of_date) - pd.Timedelta(days=45)).timestamp())
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/%5EIRX?period1={start}&period2={end}&interval=1d"
    with urllib.request.urlopen(url, timeout=20) as response:
        payload = json.loads(response.read().decode("utf-8"))
    result = payload["chart"]["result"][0]
    timestamps = result["timestamp"]
    closes = result["indicators"]["quote"][0]["close"]
    rows = [
        (pd.to_datetime(ts, unit="s"), close)
        for ts, close in zip(timestamps, closes)
        if close is not None
    ]
    if not rows:
        raise ValueError("No Yahoo ^IRX observations returned.")
    obs_date, close = rows[-1]
    return {
        "source": "Yahoo Finance",
        "series_id": "^IRX",
        "description": "13 Week Treasury Bill Yield",
        "observation_date": obs_date.date().isoformat(),
        "annual_rate_percent": float(close),
        "annual_rate_decimal": float(close) / 100.0,
    }


selected = pd.read_csv(OUTPUT_DIR / "selected_stocks.csv")
returns_all = pd.read_parquet(DATA_DIR / "returns_matrix.parquet").sort_index()

if "ticker" not in selected.columns:
    raise ValueError("selected_stocks.csv must contain a 'ticker' column.")

selected_tickers = selected["ticker"].astype(str).tolist()
missing = sorted(set(selected_tickers) - set(returns_all.columns))
if missing:
    raise ValueError(f"Selected tickers missing from returns matrix: {missing}")

returns_selected = returns_all[selected_tickers].copy()
latest_return_date = returns_selected.index.max()
if LOOKBACK_YEARS is None:
    train_returns = returns_selected.dropna(how="any")
    lookback_mode = "full_history"
else:
    train_start_date = latest_return_date - pd.DateOffset(years=LOOKBACK_YEARS)
    train_returns = returns_selected.loc[returns_selected.index > train_start_date].dropna(how="any")
    lookback_mode = f"{LOOKBACK_YEARS}_calendar_years"
train_start_date = train_returns.index.min()

if len(train_returns) < 252:
    raise ValueError("Not enough clean training observations after filtering selected stocks.")

TRADING_DAYS = estimate_trading_days_per_year(returns_all.index)

try:
    risk_free_info = fetch_fred_risk_free_rate(latest_return_date)
except Exception as fred_error:
    try:
        risk_free_info = fetch_yahoo_irx_risk_free_rate(latest_return_date)
        risk_free_info["fallback_reason"] = f"FRED fetch failed: {fred_error}"
    except Exception as yahoo_error:
        warnings.warn(
            "Could not fetch real risk-free rate from FRED or Yahoo. "
            f"FRED error: {fred_error}; Yahoo error: {yahoo_error}. Falling back to 0.0."
        )
        risk_free_info = {
            "source": "fallback",
            "series_id": "none",
            "description": "Fallback when online sources are unavailable",
            "observation_date": latest_return_date.date().isoformat(),
            "annual_rate_percent": 0.0,
            "annual_rate_decimal": 0.0,
            "fallback_reason": f"FRED error: {fred_error}; Yahoo error: {yahoo_error}",
        }

RISK_FREE_RATE = risk_free_info["annual_rate_decimal"]
RISK_FREE_RATE_DAILY = (1 + RISK_FREE_RATE) ** (1 / TRADING_DAYS) - 1
risk_free_info["daily_rate_decimal"] = RISK_FREE_RATE_DAILY
risk_free_info["as_of_returns_date"] = latest_return_date.date().isoformat()
risk_free_info["trading_days_per_year"] = TRADING_DAYS
risk_free_info["lookback_mode"] = lookback_mode
risk_free_info["lookback_years"] = "all_available" if LOOKBACK_YEARS is None else LOOKBACK_YEARS
risk_free_info["train_start_date"] = train_returns.index.min().date().isoformat()
risk_free_info["train_end_date"] = train_returns.index.max().date().isoformat()
risk_free_info["train_observations"] = len(train_returns)

pd.DataFrame([risk_free_info]).to_csv(OUTPUT_DIR / "risk_free_rate.csv", index=False)
pd.DataFrame([{
    "trading_days_per_year": TRADING_DAYS,
    "lookback_mode": lookback_mode,
    "lookback_years": "all_available" if LOOKBACK_YEARS is None else LOOKBACK_YEARS,
    "train_start_date": train_returns.index.min().date().isoformat(),
    "train_end_date": train_returns.index.max().date().isoformat(),
    "train_observations": len(train_returns),
    "risk_free_rate_annual_decimal": RISK_FREE_RATE,
    "risk_free_rate_daily_decimal": RISK_FREE_RATE_DAILY,
    "risk_free_source": risk_free_info["source"],
    "risk_free_series_id": risk_free_info["series_id"],
    "risk_free_observation_date": risk_free_info["observation_date"],
}]).to_csv(OUTPUT_DIR / "portfolio_allocation_inputs.csv", index=False)

selected_info_cols = [
    col for col in ["ticker", "company_name", "sector", "sub_industry", "cluster_id", "sharpe_ratio"]
    if col in selected.columns
]
selected_info = selected[selected_info_cols].copy()

print(f"Selected stocks: {len(selected_tickers)}")
print(f"Full returns matrix: {returns_selected.shape[0]} days x {returns_selected.shape[1]} stocks")
print(f"Training window: {train_returns.index.min().date()} to {train_returns.index.max().date()}")
print(f"Training observations used: {len(train_returns)}")
print(f"Lookback mode: {lookback_mode}")
print(f"Estimated trading days/year from data: {TRADING_DAYS}")
print(
    f"Risk-free rate: {RISK_FREE_RATE:.4%} annual "
    f"from {risk_free_info['source']} {risk_free_info['series_id']} "
    f"observed {risk_free_info['observation_date']}"
)
selected_info.head()

Selected stocks: 25
Full returns matrix: 1842 days x 25 stocks
Training window: 2019-01-03 to 2026-05-01
Training observations used: 1842
Lookback mode: full_history
Estimated trading days/year from data: 252
Risk-free rate: 3.6800% annual from FRED DGS3MO observed 2026-05-01


In [3]:
# ----------------------------------------------------------------------
# Shared helper functions
# ----------------------------------------------------------------------
def annualized_mean(returns: pd.DataFrame) -> pd.Series:
    return returns.mean() * TRADING_DAYS


def annualized_cov(returns: pd.DataFrame) -> pd.DataFrame:
    cov = returns.cov() * TRADING_DAYS
    jitter = 1e-10
    return cov + np.eye(cov.shape[0]) * jitter


def cap_and_redistribute(weights, max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT, tol=1e-12):
    # Normalize weights, cap concentration, and redistribute excess to uncapped names.
    w = np.asarray(weights, dtype=float).copy()
    n = len(w)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.maximum(w, min_weight)

    if w.sum() <= tol:
        w = np.ones(n) / n
    else:
        w = w / w.sum()

    if max_weight is None:
        return w

    if n * max_weight < 1 - 1e-10:
        raise ValueError("MAX_WEIGHT is too low to allocate 100% across the selected stocks.")

    for _ in range(100):
        over = w > max_weight + tol
        if not over.any():
            break
        excess = (w[over] - max_weight).sum()
        w[over] = max_weight
        under = w < max_weight - tol
        if not under.any():
            break
        capacity = max_weight - w[under]
        if capacity.sum() <= tol:
            break
        add = np.minimum(capacity, excess * capacity / capacity.sum())
        w[under] += add

    return w / w.sum()


def portfolio_metrics(weights, returns: pd.DataFrame, alpha=CVAR_ALPHA):
    w = pd.Series(weights, index=returns.columns)
    port_ret = returns @ w
    ann_ret = port_ret.mean() * TRADING_DAYS
    ann_vol = port_ret.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = np.nan if ann_vol == 0 else (ann_ret - RISK_FREE_RATE) / ann_vol
    var = -port_ret.quantile(1 - alpha)
    cvar = -port_ret[port_ret <= port_ret.quantile(1 - alpha)].mean()
    effective_n = 1 / np.sum(np.square(w.values))
    return {
        "annual_return": ann_ret,
        "annual_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "historical_var_95_daily": var,
        "historical_cvar_95_daily": cvar,
        "max_weight": w.max(),
        "min_weight": w.min(),
        "effective_number_of_holdings": effective_n,
    }


def export_weights(method_name, weights, selected_info=selected_info):
    out = pd.DataFrame({"ticker": train_returns.columns, "weight": weights})
    out = out.merge(selected_info, on="ticker", how="left")
    out = out.sort_values("weight", ascending=False)
    file_name = f"portfolio_weights_{method_name}.csv"
    out.to_csv(OUTPUT_DIR / file_name, index=False)
    return out

## Markowitz-style expected return limitation

Expected returns are estimated from historical mean returns. As a result, portfolios with very high annualized expected return may be driven by assets that strongly outperformed during the sample period. The results should therefore be interpreted as allocation comparisons rather than precise forecasts of future performance.


## CVaR scenario assumptions

Bootstrap scenarios resample historical daily return rows, so they preserve empirical co-movement across assets and include historical tail events observed in the sample. Monte Carlo scenarios are generated from a multivariate normal distribution using historical mean returns and the historical covariance matrix.

Bootstrap is usually more realistic for empirical tail-risk analysis because it relies less on a parametric return distribution. Monte Carlo is useful as a parametric robustness check, but it may underestimate extreme losses when equity returns have fat tails, skewness, or regime shifts.


In [4]:
# ----------------------------------------------------------------------
# Scenario generation for CVaR
# ----------------------------------------------------------------------
def generate_bootstrap_scenarios(returns: pd.DataFrame, n_scenarios=N_SCENARIOS, seed=RANDOM_SEED):
    # Sample whole historical return rows with replacement to preserve cross-asset co-movement.
    rng = np.random.default_rng(seed)
    values = returns.to_numpy()
    idx = rng.integers(0, len(values), size=n_scenarios)
    return values[idx]


def generate_monte_carlo_scenarios(returns: pd.DataFrame, n_scenarios=N_SCENARIOS, seed=RANDOM_SEED):
    # Sample daily asset returns from a multivariate normal estimated from historical mean/covariance.
    rng = np.random.default_rng(seed)
    mu = returns.mean().to_numpy()
    cov = returns.cov().to_numpy()
    cov = cov + np.eye(cov.shape[0]) * 1e-10
    return rng.multivariate_normal(mean=mu, cov=cov, size=n_scenarios)


bootstrap_scenarios = generate_bootstrap_scenarios(train_returns)
monte_carlo_scenarios = generate_monte_carlo_scenarios(train_returns)

print("Bootstrap scenarios shape:", bootstrap_scenarios.shape)
print("Monte Carlo scenarios shape:", monte_carlo_scenarios.shape)

Bootstrap scenarios shape: (3000, 25)
Monte Carlo scenarios shape: (3000, 25)


In [5]:
# ----------------------------------------------------------------------
# Allocation models
# ----------------------------------------------------------------------
def allocate_equal_weight(returns: pd.DataFrame):
    n = returns.shape[1]
    return np.ones(n) / n


def allocate_inverse_volatility(returns: pd.DataFrame):
    vol = returns.std(ddof=1).replace(0, np.nan)
    inv_vol = 1 / vol
    inv_vol = inv_vol.replace([np.inf, -np.inf], np.nan).fillna(0)
    return cap_and_redistribute(inv_vol.to_numpy())


def allocate_markowitz(returns: pd.DataFrame, delta: float):
    mu = annualized_mean(returns).to_numpy()
    cov = annualized_cov(returns).to_numpy()
    n = len(mu)

    def objective(w):
        portfolio_vol = np.sqrt(max(w @ cov @ w, 0.0))
        return -(mu @ w - delta * portfolio_vol)

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = [(MIN_WEIGHT, MAX_WEIGHT) for _ in range(n)]
    x0 = np.ones(n) / n

    result = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 1000})
    if not result.success:
        warnings.warn(f"Markowitz optimizer failed: {result.message}. Falling back to equal weight.")
        return allocate_equal_weight(returns)
    return cap_and_redistribute(result.x)


def allocate_risk_parity(returns: pd.DataFrame):
    cov = annualized_cov(returns).to_numpy()
    n = cov.shape[0]
    target_budget = np.ones(n) / n

    def risk_contributions(w):
        portfolio_var = w @ cov @ w
        if portfolio_var <= 0:
            return np.ones(n) / n
        marginal_risk = cov @ w
        return w * marginal_risk / portfolio_var

    def objective(w):
        rc = risk_contributions(w)
        return np.sum((rc - target_budget) ** 2)

    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
    bounds = [(MIN_WEIGHT, MAX_WEIGHT) for _ in range(n)]
    x0 = allocate_inverse_volatility(returns)

    result = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 2000})
    if not result.success:
        warnings.warn(f"Risk Parity optimizer failed: {result.message}. Falling back to inverse volatility.")
        return allocate_inverse_volatility(returns)
    return cap_and_redistribute(result.x)


def allocate_cvar(scenarios: np.ndarray, fallback_returns: pd.DataFrame, alpha=CVAR_ALPHA, return_tradeoff=CVAR_RETURN_TRADEOFF):
    # Mean-CVaR linear program using scenario returns.
    # Variables: weights w, VaR threshold t, tail excess losses u.
    # Minimize: CVaR(loss) - return_tradeoff * expected_return.
    scenarios = np.asarray(scenarios, dtype=float)
    n_scenarios, n_assets = scenarios.shape
    scenario_mean = scenarios.mean(axis=0)

    n_vars = n_assets + 1 + n_scenarios
    t_idx = n_assets
    u_start = n_assets + 1

    c = np.zeros(n_vars)
    c[:n_assets] = -return_tradeoff * scenario_mean
    c[t_idx] = 1.0
    c[u_start:] = 1.0 / ((1.0 - alpha) * n_scenarios)

    # Constraint: -R_s @ w - t - u_s <= 0 for every scenario s
    A_ub = np.zeros((n_scenarios, n_vars))
    A_ub[:, :n_assets] = -scenarios
    A_ub[:, t_idx] = -1.0
    A_ub[np.arange(n_scenarios), u_start + np.arange(n_scenarios)] = -1.0
    b_ub = np.zeros(n_scenarios)

    A_eq = np.zeros((1, n_vars))
    A_eq[0, :n_assets] = 1.0
    b_eq = np.array([1.0])

    bounds = [(MIN_WEIGHT, MAX_WEIGHT) for _ in range(n_assets)] + [(None, None)] + [(0.0, None) for _ in range(n_scenarios)]

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not result.success:
        warnings.warn(f"CVaR optimizer failed: {result.message}. Falling back to inverse volatility on the same asset universe.")
        return allocate_inverse_volatility(fallback_returns)

    return cap_and_redistribute(result.x[:n_assets])

In [6]:
# ----------------------------------------------------------------------
# Run all portfolio allocation methods
# ----------------------------------------------------------------------
def scenario_display_name(scenario_name):
    return {"bootstrap": "Bootstrap", "montecarlo": "Monte Carlo"}.get(scenario_name, scenario_name.title())


def tradeoff_label(value):
    label = f"{value:g}".replace(".", "_")
    return f"t{label}"


def delta_label(value):
    label = f"{value:.4g}".replace(".", "_")
    return f"d{label}"


def scenario_cvar_metrics(weights, scenarios, alpha=CVAR_ALPHA):
    port_returns = np.asarray(scenarios) @ np.asarray(weights)
    threshold = np.quantile(port_returns, 1 - alpha)
    tail = port_returns[port_returns <= threshold]
    return {
        "scenario_mean_daily": float(port_returns.mean()),
        "scenario_var_95_daily": float(-threshold),
        "scenario_cvar_95_daily": float(-tail.mean()),
    }


def build_markowitz_frontier():
    frontier_weights = {}
    frontier_rows = []

    for delta in MARKOWITZ_DELTA_GRID:
        method = f"markowitz_{delta_label(delta)}"
        w = allocate_markowitz(train_returns, delta=delta)
        frontier_weights[method] = w

        metrics = portfolio_metrics(w, train_returns)
        metrics["method"] = method
        metrics["delta"] = delta
        metrics["penalty"] = "annual_volatility"
        metrics["weight_sum"] = float(np.sum(w))
        frontier_rows.append(metrics)

    return frontier_weights, frontier_rows


def build_cvar_frontier(scenario_name, scenarios):
    frontier_weights = {}
    frontier_rows = []

    for tradeoff in CVAR_TRADEOFF_GRID:
        method = f"cvar_{scenario_name}_{tradeoff_label(tradeoff)}"
        w = allocate_cvar(scenarios, fallback_returns=train_returns, return_tradeoff=tradeoff)
        frontier_weights[method] = w

        metrics = portfolio_metrics(w, train_returns)
        metrics.update(scenario_cvar_metrics(w, scenarios))
        metrics["method"] = method
        metrics["scenario_type"] = scenario_name
        metrics["Scenario Type"] = scenario_display_name(scenario_name)
        metrics["Risk Metric"] = "CVaR 95%"
        metrics["Horizon"] = "Daily"
        metrics["Portfolio Type"] = "CVaR Optimized"
        metrics["return_tradeoff"] = tradeoff
        metrics["weight_sum"] = float(np.sum(w))
        frontier_rows.append(metrics)

    return frontier_weights, frontier_rows


# Non-scenario allocation methods.
base_weights = {
    "equal": allocate_equal_weight(train_returns),
    "inverse_volatility": allocate_inverse_volatility(train_returns),
    "risk_parity": allocate_risk_parity(train_returns),
}

# Markowitz-style efficient frontier.
markowitz_frontier_weights, markowitz_frontier_rows = build_markowitz_frontier()
markowitz_frontier = pd.DataFrame(markowitz_frontier_rows).set_index("method")
markowitz_frontier.to_csv(OUTPUT_DIR / "portfolio_markowitz_frontier_summary.csv")

markowitz_frontier_weights_table = pd.DataFrame(markowitz_frontier_weights, index=train_returns.columns)
markowitz_frontier_weights_table.index.name = "ticker"
markowitz_frontier_weights_table.to_csv(OUTPUT_DIR / "portfolio_markowitz_frontier_weights_all.csv")

for method, w in markowitz_frontier_weights.items():
    export_weights(method, w)

# CVaR efficient frontiers.
bootstrap_frontier_weights, bootstrap_frontier_rows = build_cvar_frontier("bootstrap", bootstrap_scenarios)
montecarlo_frontier_weights, montecarlo_frontier_rows = build_cvar_frontier("montecarlo", monte_carlo_scenarios)

cvar_frontier_weights = {**bootstrap_frontier_weights, **montecarlo_frontier_weights}
cvar_frontier = pd.DataFrame(bootstrap_frontier_rows + montecarlo_frontier_rows).set_index("method")
cvar_frontier.to_csv(OUTPUT_DIR / "portfolio_cvar_frontier_summary.csv")

cvar_frontier_weights_table = pd.DataFrame(cvar_frontier_weights, index=train_returns.columns)
cvar_frontier_weights_table.index.name = "ticker"
cvar_frontier_weights_table.to_csv(OUTPUT_DIR / "portfolio_cvar_frontier_weights_all.csv")

for method, w in cvar_frontier_weights.items():
    export_weights(method, w)

# Keep a headline comparison table. The Markowitz-style method is represented by the max-training-Sharpe point on the mean-volatility frontier.
def closest_grid_label(grid, target, label_func):
    chosen = min(grid, key=lambda x: abs(np.log(x) - np.log(target)))
    return label_func(chosen)

markowitz_default_method = markowitz_frontier["sharpe_ratio"].idxmax()
markowitz_headline_method = "markowitz_best_sharpe_default"
cvar_balanced_label = closest_grid_label(CVAR_TRADEOFF_GRID, CVAR_RETURN_TRADEOFF, tradeoff_label)

weights = {
    **base_weights,
    markowitz_headline_method: markowitz_frontier_weights[markowitz_default_method],
    "cvar_bootstrap": cvar_frontier_weights[f"cvar_bootstrap_{cvar_balanced_label}"],
    "cvar_montecarlo": cvar_frontier_weights[f"cvar_montecarlo_{cvar_balanced_label}"],
}

weights_table = pd.DataFrame(weights, index=train_returns.columns)
weights_table.index.name = "ticker"
weights_table.to_csv(OUTPUT_DIR / "portfolio_weights_all.csv")

summary_rows = []
for method, w in weights.items():
    export_weights(method, w)
    metrics = portfolio_metrics(w, train_returns)
    metrics["method"] = method
    metrics["weight_sum"] = float(np.sum(w))
    summary_rows.append(metrics)

summary = pd.DataFrame(summary_rows).set_index("method")
summary.to_csv(OUTPUT_DIR / "portfolio_allocation_summary.csv")

print("Saved portfolio weights, Markowitz/CVaR frontiers, and allocation summary.")
print(f"Default Markowitz frontier point in headline table: {markowitz_headline_method} (source={markowitz_default_method})")
print(f"Default CVaR tradeoff in headline table: {CVAR_RETURN_TRADEOFF} ({cvar_balanced_label})")
summary.round(4)

Saved portfolio weights, Markowitz/CVaR frontiers, and allocation summary.
Default Markowitz frontier point in headline table: markowitz_best_sharpe_default (source=markowitz_d2_154)
Default CVaR tradeoff in headline table: 1.0 (t1)


In [7]:
# ----------------------------------------------------------------------
# Display top weights by method
# ----------------------------------------------------------------------
for method in weights_table.columns:
    print("\n" + "=" * 80)
    print(method)
    print("=" * 80)
    display(weights_table[method].sort_values(ascending=False).head(10).to_frame("weight"))

print("\n" + "=" * 80)
print("Markowitz-style mean-volatility frontier summary")
print("=" * 80)
display(markowitz_frontier[[
    "delta",
    "penalty",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "historical_cvar_95_daily",
    "effective_number_of_holdings",
]].round(4))

print("\n" + "=" * 80)
print("CVaR efficient frontier summary")
print("=" * 80)
display(cvar_frontier[[
    "Scenario Type",
    "Risk Metric",
    "Horizon",
    "Portfolio Type",
    "scenario_type",
    "return_tradeoff",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "historical_cvar_95_daily",
    "scenario_cvar_95_daily",
    "effective_number_of_holdings",
]].round(4))


equal
        weight
ticker        
PM        0.04
TGT       0.04
COST      0.04
KR        0.04
IRM       0.04
MCD       0.04
TMUS      0.04
CBOE      0.04
PGR       0.04
STX       0.04

inverse_volatility
          weight
ticker          
MCD     0.060638
COST    0.057631
PM      0.051455
PGR     0.049935
CBOE    0.049735
TMUS    0.048995
MSI     0.048995
KR      0.046563
EA      0.046418
MCK     0.045792

risk_parity
          weight
ticker          
KR      0.070272
CBOE    0.057069
EA      0.051481
NEM     0.050047
PGR     0.049745
MCD     0.049031
PM      0.047274
COST    0.045888
SW      0.044948
MCK     0.044863

markowitz_best_sharpe_default
          weight
ticker          
KR      0.100000
CBOE    0.100000
MCK     0.100000
LLY     0.100000
PWR     0.100000
STX     0.082201
PGR     0.080702
NVDA    0.073079
NEM     0.072641
COST    0.062885

cvar_bootstrap
          weight
ticker          
PM      0.100000
KR      0.100000
PGR     0.100000
CBOE    0.100000
NEM     0.100000
EA

In [8]:
# ----------------------------------------------------------------------
# Save a compact weight heatmap
# ----------------------------------------------------------------------
plot_data = weights_table.T[weights_table.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(plot_data.values, aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(plot_data.shape[1]))
ax.set_xticklabels(plot_data.columns, rotation=90)
ax.set_yticks(range(plot_data.shape[0]))
ax.set_yticklabels(plot_data.index)
ax.set_title("Portfolio Weights by Allocation Method")
fig.colorbar(im, ax=ax, label="Weight")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "portfolio_weights_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

C:\Users\oakza\Documents\stock-selection-funnel\notebooks\03_allocate_portfolios.ipynb#cell-10:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [9]:
# ----------------------------------------------------------------------
# Save Markowitz efficient frontier chart
# ----------------------------------------------------------------------
data = markowitz_frontier.sort_values("delta")

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    data["annual_volatility"],
    data["annual_return"],
    c=data["delta"],
    cmap="plasma_r",
    s=100,
    edgecolor="black",
)
label_positions = np.unique(np.linspace(0, len(data) - 1, min(8, len(data)), dtype=int))
for idx in label_positions:
    row = data.iloc[idx]
    ax.annotate(
        f"{row['delta']:.4g}",
        (row["annual_volatility"], row["annual_return"]),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=8,
    )
ax.set_title("Markowitz-style Mean-Volatility Frontier")
ax.set_xlabel("Annual Volatility")
ax.set_ylabel("Annual Return")
ax.grid(alpha=0.3)
fig.colorbar(scatter, ax=ax, label="Delta")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "portfolio_markowitz_efficient_frontier.png", dpi=200, bbox_inches="tight")
plt.show()

C:\Users\oakza\Documents\stock-selection-funnel\notebooks\03_allocate_portfolios.ipynb#cell-11:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### CVaR risk-return plots

These plots keep the standard risk-return axes and use color to show the scenario CVaR estimate.

In [ ]:
scenario_order = ["bootstrap", "montecarlo"]
scenario_titles = {"bootstrap": "Bootstrap", "montecarlo": "Monte Carlo"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, scenario_type in zip(axes, scenario_order):
    data = cvar_frontier[cvar_frontier["scenario_type"] == scenario_type].sort_values("return_tradeoff")
    scatter = ax.scatter(
        data["annual_volatility"],
        data["annual_return"],
        c=data["scenario_cvar_95_daily"],
        cmap="viridis_r",
        s=80,
        edgecolor="black",
    )
    label_positions = np.unique(np.linspace(0, len(data) - 1, min(8, len(data)), dtype=int))
    for idx in label_positions:
        row = data.iloc[idx]
        ax.annotate(
            f"{row['return_tradeoff']:.3g}",
            (row["annual_volatility"], row["annual_return"]),
            textcoords="offset points",
            xytext=(6, 5),
            fontsize=8,
        )
    ax.set_title(f"Risk-Return Plot Colored by CVaR - {scenario_titles[scenario_type]}")
    ax.set_xlabel("Annual Volatility")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Annual Return")
fig.colorbar(scatter, ax=axes, label="Scenario CVaR 95% Daily")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "portfolio_cvar_efficient_frontier.png", dpi=200, bbox_inches="tight")
plt.show()

### Mean-CVaR frontier plots

These plots put scenario CVaR directly on the x-axis, so the return versus tail-risk trade-off is easier to explain.

In [ ]:
scenario_order = ["bootstrap", "montecarlo"]
scenario_titles = {"bootstrap": "Bootstrap", "montecarlo": "Monte Carlo"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, scenario_type in zip(axes, scenario_order):
    data = cvar_frontier[cvar_frontier["scenario_type"] == scenario_type].sort_values("scenario_cvar_95_daily")
    scatter = ax.scatter(
        data["scenario_cvar_95_daily"],
        data["annual_return"],
        c=data["annual_volatility"],
        cmap="magma_r",
        s=80,
        edgecolor="black",
    )
    label_positions = np.unique(np.linspace(0, len(data) - 1, min(8, len(data)), dtype=int))
    for idx in label_positions:
        row = data.iloc[idx]
        ax.annotate(
            f"{row['return_tradeoff']:.3g}",
            (row["scenario_cvar_95_daily"], row["annual_return"]),
            textcoords="offset points",
            xytext=(6, 5),
            fontsize=8,
        )
    ax.set_title(f"Mean-CVaR Frontier - {scenario_titles[scenario_type]}")
    ax.set_xlabel("Scenario CVaR 95% Daily")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Annual Return")
fig.colorbar(scatter, ax=axes, label="Annual Volatility")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "portfolio_mean_cvar_frontier.png", dpi=200, bbox_inches="tight")
plt.show()

## Outputs

This step writes portfolio weights, allocation summaries, Markowitz-style frontier files, CVaR frontier files, risk-contribution diagnostics, and comparison charts to `outputs/`.

## Additional Diagnostic Charts

These charts compare risk-return position, headline metrics, top holdings, risk contributions, and CVaR scenario return distributions.

### Diagnostic chart setup

The following charts use the saved allocation tables, so they are interpretation checks after the allocation models have already run.

In [ ]:
# ----------------------------------------------------------------------
# Additional diagnostic charts for portfolio allocation
# ----------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")

def short_method_name(method):
    return {
        "equal": "Equal",
        "inverse_volatility": "Inv Vol",
        "risk_parity": "Risk Parity",
        "markowitz_best_sharpe_default": "Mean-Vol",
        "cvar_bootstrap": "CVaR Boot",
        "cvar_montecarlo": "CVaR MC",
    }.get(method, method)

# Reload compact outputs so this cell can be rerun independently after the main notebook has produced outputs.
summary_chart = pd.read_csv(OUTPUT_DIR / "portfolio_allocation_summary.csv", index_col=0)
weights_chart = pd.read_csv(OUTPUT_DIR / "portfolio_weights_all.csv", index_col=0)

returns_for_charts = returns_selected.dropna(how="any")
if LOOKBACK_YEARS is not None:
    chart_start = returns_for_charts.index.max() - pd.DateOffset(years=LOOKBACK_YEARS)
    returns_for_charts = returns_for_charts.loc[returns_for_charts.index > chart_start]

### Portfolio allocation risk-return comparison

This chart compares the headline allocation methods after all weights have been produced.

In [ ]:
# 1) Risk-return scatter: easiest view of the tradeoff each allocation makes.
fig, ax = plt.subplots(figsize=(9, 6))
points = ax.scatter(
    summary_chart["annual_volatility"],
    summary_chart["annual_return"],
    c=summary_chart["sharpe_ratio"],
    s=170,
    cmap="viridis",
    edgecolor="black",
    linewidth=0.7,
)
for method, row in summary_chart.iterrows():
    ax.annotate(
        short_method_name(method),
        (row["annual_volatility"], row["annual_return"]),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=9,
    )
ax.set_title("Portfolio Allocation Risk vs Return")
ax.set_xlabel("Annualized Volatility")
ax.set_ylabel("Annualized Return")
fig.colorbar(points, ax=ax, label="Sharpe Ratio")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_allocation_risk_return_scatter.png", dpi=160)
plt.show()

### Allocation metric dashboard

The dashboard puts return, volatility, Sharpe ratio, and historical daily CVaR next to each other for quick comparison.

In [ ]:
# 2) Metric dashboard: return, volatility, Sharpe, and historical CVaR side by side.
metric_specs = [
    ("annual_return", "Annual Return", False),
    ("annual_volatility", "Annual Volatility", False),
    ("sharpe_ratio", "Sharpe Ratio", False),
    ("historical_cvar_95_daily", "Historical CVaR 95% Daily", False),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, title, invert) in zip(axes.ravel(), metric_specs):
    values = summary_chart[col].sort_values(ascending=invert)
    ax.bar([short_method_name(x) for x in values.index], values.values, color="#4C78A8")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_allocation_metric_bars.png", dpi=160)
plt.show()

### Top holdings by allocation method

This chart explains why optimizer-driven portfolios can behave differently from Equal Weight.

In [ ]:
# 3) Top holdings per method: shows why equal weight differs from optimizer-driven portfolios.
methods = list(weights_chart.columns)
fig, axes = plt.subplots(3, 2, figsize=(13, 12))
for ax, method in zip(axes.ravel(), methods):
    top = weights_chart[method].sort_values(ascending=True).tail(10)
    ax.barh(top.index, top.values, color="#59A14F")
    ax.set_title(short_method_name(method))
    ax.set_xlim(0, max(MAX_WEIGHT * 1.08, top.max() * 1.15))
    ax.axvline(MAX_WEIGHT, color="#E15759", linestyle="--", linewidth=1, alpha=0.8)
for ax in axes.ravel()[len(methods):]:
    ax.axis("off")
fig.suptitle("Top 10 Weights by Allocation Method", y=0.995)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_top10_weights_by_method.png", dpi=160)
plt.show()

### Signed and absolute risk contribution heatmaps

Negative risk contribution can occur when an asset has negative or very low covariance with the overall portfolio, meaning it reduces total portfolio variance.

In [ ]:
# 4) Risk contribution heatmap: useful for checking Risk Parity and concentration of portfolio risk.
cov_chart = returns_for_charts.cov() * TRADING_DAYS + np.eye(returns_for_charts.shape[1]) * 1e-10
risk_contrib = {}
for method in methods:
    w = weights_chart[method].reindex(returns_for_charts.columns).fillna(0.0).to_numpy()
    marginal = cov_chart.to_numpy() @ w
    portfolio_var = float(w @ marginal)
    if portfolio_var <= 0:
        rc = np.zeros_like(w)
    else:
        rc = w * marginal / portfolio_var
    risk_contrib[method] = rc

risk_contrib_table = pd.DataFrame(risk_contrib, index=returns_for_charts.columns)
risk_contrib_table.index.name = "ticker"
risk_contrib_table.to_csv(OUTPUT_DIR / "portfolio_risk_contributions.csv")

ordered_tickers = risk_contrib_table.abs().max(axis=1).sort_values(ascending=False).index
heat = risk_contrib_table.loc[ordered_tickers, methods].T
max_abs = float(np.nanmax(np.abs(heat.to_numpy())))
fig, ax = plt.subplots(figsize=(13, 5.8))
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-max_abs, vmax=max_abs)
ax.set_yticks(np.arange(len(methods)))
ax.set_yticklabels([short_method_name(m) for m in methods])
ax.set_xticks(np.arange(len(ordered_tickers)))
ax.set_xticklabels(ordered_tickers, rotation=90)
ax.set_title("Portfolio Risk Contribution by Ticker")
fig.colorbar(im, ax=ax, label="Share of Portfolio Variance")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_risk_contribution_heatmap.png", dpi=160)
plt.show()

### Absolute normalized risk contribution heatmap

This version ignores the sign and normalizes each row, making contribution magnitudes easier to compare across methods.

In [ ]:
abs_heat = heat.abs()
row_sums = abs_heat.sum(axis=1).replace(0, np.nan)
abs_heat_pct = abs_heat.div(row_sums, axis=0).fillna(0.0)
fig, ax = plt.subplots(figsize=(13, 5.8))
im = ax.imshow(abs_heat_pct.to_numpy(), aspect="auto", cmap="Blues")
ax.set_yticks(np.arange(len(methods)))
ax.set_yticklabels([short_method_name(m) for m in methods])
ax.set_xticks(np.arange(len(ordered_tickers)))
ax.set_xticklabels(ordered_tickers, rotation=90)
ax.set_title("Absolute Normalized Risk Contribution by Ticker")
fig.colorbar(im, ax=ax, label="Share of Absolute Risk Contribution")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_absolute_risk_contribution_heatmap.png", dpi=160)
plt.show()

### CVaR scenario return distributions

This diagnostic checks the simulated portfolio-return distributions used to interpret the CVaR Bootstrap and CVaR Monte Carlo allocations.

In [ ]:
# 5) CVaR scenario distributions: shows what the two scenario engines feed into CVaR.
bootstrap_for_chart = generate_bootstrap_scenarios(returns_for_charts, n_scenarios=N_SCENARIOS, seed=RANDOM_SEED)
monte_carlo_for_chart = generate_monte_carlo_scenarios(returns_for_charts, n_scenarios=N_SCENARIOS, seed=RANDOM_SEED)
scenario_series = {
    "CVaR Bootstrap": bootstrap_for_chart @ weights_chart["cvar_bootstrap"].reindex(returns_for_charts.columns).to_numpy(),
    "CVaR Monte Carlo": monte_carlo_for_chart @ weights_chart["cvar_montecarlo"].reindex(returns_for_charts.columns).to_numpy(),
}
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in scenario_series.items():
    ax.hist(values, bins=70, alpha=0.45, density=True, label=label)
    tail_cutoff = np.quantile(values, 1 - CVAR_ALPHA)
    ax.axvline(tail_cutoff, linestyle="--", linewidth=1.2, label=f"{label} 5% tail")
ax.set_title("CVaR Scenario Portfolio Return Distributions")
ax.set_xlabel("Daily Portfolio Return Scenario")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "portfolio_cvar_scenario_return_distributions.png", dpi=160)
plt.show()

print("Saved additional allocation diagnostic charts.")